# 00 · Capture the main draw (GPU, about 7 h on an A100)

Generates the candidates and hidden-state features behind **Tables I and II**
(selection and reading-depth ablation).

* Generator: frozen `Qwen/Qwen2.5-1.5B` (base), four worked exemplars,
  temperature 0.7, top-p 0.95, 256 new tokens, **K = 8**.
* Verifier pool: **all 7,473 GSM8K training questions** (59,784 candidates).
* Test: all 1,319 GSM8K test questions (10,552 candidates).
* Features per candidate: mean-pooled hidden states at layers 14, 21, 28
  (plus last-token and answer-span variants) and the five scalars z(c).

Both captures are resumable: rerun the cell after a disconnect. Sampling on GPU
is not bit-reproducible, so a fresh capture is a new random draw; to reproduce
the exact published numbers, use the released caches and skip to notebook 02.

In [ ]:
#@title Setup: mount Drive, install lvpp, set cache paths
import os, sys, json, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                    "lvpp[gpu] @ git+https://github.com/KARTIKPANSURIYA/lvpp.git"], check=True)

# Root of the cached candidates/features. The default is the Drive layout the
# paper's runs wrote to; set LVPP_CACHE to use another location.
CACHE = os.environ.get("LVPP_CACHE", "/content/drive/MyDrive/lcldd_runs")
RESULTS = os.environ.get("LVPP_RESULTS", os.path.join(CACHE, "lvpp_results"))
os.makedirs(RESULTS, exist_ok=True)
PATHS = dict(
    main_pool   = f"{CACHE}/verifier_upgrade/pool.pt",       # 7,473 GSM8K train questions x 8
    main_test   = f"{CACHE}/verifier_upgrade/testfeat.pt",   # 1,319 GSM8K test questions x 8
    prefix_root = f"{CACHE}/abort_main_draw",                # prefix draw: pool/ and test/ shards
    span_root   = f"{CACHE}/span_fixed_vllm",                # Fig. 2 draw (1.5B-Instruct, corrected trigger)
    mi_pool     = f"{CACHE}/pending/mi_pool.pt",             # Math-Instruct on GSM8K
    mi_test     = f"{CACHE}/pending/mi_test.pt",
    math_pool   = f"{CACHE}/pending/math_pool.pt",           # Math-Instruct on MATH
    math_test   = f"{CACHE}/pending/math_test.pt",
    x15_pool    = f"{CACHE}/novelty/x15_pool.pt",            # 1.5B-Instruct, 1,200 questions
    x3_pool     = f"{CACHE}/novelty/x3_pool.pt",             # 3B-Instruct, 1,200 questions
    x3_test     = f"{CACHE}/novelty/x3_test.pt",             # 3B-Instruct, 400 test questions
    legacy_fewshot = f"{CACHE}/verifier_fewshot",            # Table IV, single-layer verifier
    legacy_mathmax = f"{CACHE}/verifier_mathmax",
    distilled_r32  = f"{CACHE}/verifier_final",
)
import numpy as np, pandas as pd, torch
from sklearn.metrics import roc_auc_score
print("device:", "cuda" if torch.cuda.is_available() else "cpu", "| cache:", CACHE)

In [ ]:
def report(rows, expected, keys, tol=0.6):
    """Print measured vs published values. Regenerated candidates are a new
    random draw, so agreement within about half a point is expected; from the
    released caches the numbers should match to the reported precision."""
    out = []
    for k in keys:
        m, e = rows.get(k), expected.get(k)
        if m is None or e is None:
            flag = ""
        elif e < 1e-3:                       # p-values: significant at the same level
            flag = "ok" if m < 1e-3 else "CHECK"
        elif e <= 1.0:                       # AUCs and rates
            flag = "ok" if abs(m - e) <= 0.01 else "CHECK"
        else:                                # accuracies in percentage points
            flag = "ok" if abs(m - e) <= tol else "CHECK"
        out.append(dict(quantity=k, measured=None if m is None else round(m, 4), paper=e, status=flag))
    df = pd.DataFrame(out)
    print(df.to_string(index=False))
    return df

In [ ]:
from lvpp.capture import FewShotCapture, FewShotConfig
from lvpp.data import gsm8k
os.makedirs(os.path.dirname(PATHS["main_pool"]), exist_ok=True)
cap = FewShotCapture(FewShotConfig())
train, test = gsm8k("train"), gsm8k("test")
print(len(train), "train |", len(test), "test")

In [ ]:
pool = cap.build_pool(train, PATHS["main_pool"], n_pool=7473)
print("pool:", pool["n_done"], "questions,", len(pool["labels"]), "candidates, pos-rate",
      round(float(np.mean(pool["labels"])), 3), "(paper run: 7473 q, 59784 cands, 0.768)")

In [ ]:
tf = cap.build_test(test, PATHS["main_test"])
print("test questions:", len(tf))